# Arabic PP-OCRv5 mobile recognition: direct Kaggle fine-tuning

Enable **Internet** and a GPU (T4 x2 supported). Attach `T2.zip` (or its Kaggle-extracted images), `ocr_ocr_labeling.json`, and `egyptian_car_plates_filtered_no_forbidden_classes_or_0.zip` (or its Kaggle-extracted COCO folders) as Kaggle inputs. The old JSON may instead be inside `T2.zip`. The 17 letter codes become Arabic letters; digits remain ASCII. Only annotated images train the recognizer. Old and new train splits train the model; the new valid split is validation, and both test splits are held out. Overlapping train images are excluded. Each run trains 20 epochs. To continue, attach the most recent `v5_arabic_mobile-resume-epoch-N.zip` alongside the original inputs. Training prints each step and a loss/validation summary after every epoch. Run all cells and download the exported model, resume ZIP, `training.log`, `epoch-metrics.csv`, `epoch-metrics.json`, and `training-report.md` from **Kaggle Output → /kaggle/working**. Final test contains 8 labeled old images and 156 new images.


In [ ]:
MODEL = 'v5_arabic_mobile'
CONFIG_PATH = 'configs/rec/PP-OCRv5/multi_language/arabic_PP-OCRv5_mobile_rec.yaml'
import os, sys, pathlib, subprocess, shutil
from IPython.display import display, FileLink
import yaml
WORK = pathlib.Path('/kaggle/working')
INPUT = pathlib.Path('/kaggle/input')
assert WORK.is_dir() and INPUT.is_dir(), 'Run this notebook on Kaggle'
os.chdir(WORK)  # Relative output links work in the notebook file browser.
subprocess.run(['nvidia-smi'], check=True)
# The same pinned PaddleOCR source and CUDA wheel as the IRIS Kaggle reference notebook.
subprocess.run('curl -LsSf https://astral.sh/uv/install.sh | sh', shell=True, check=True)
uv = shutil.which('uv') or '/root/.local/bin/uv'
subprocess.run([uv, 'python', 'install', '3.11'], check=True)
envdir = WORK / 'direct-ppocr-env'
if not (envdir / 'bin/python').is_file():
    subprocess.run([uv, 'venv', '--python', '3.11', '--seed', str(envdir)], check=True)
PY = str(envdir / 'bin/python')
subprocess.run([uv, 'pip', 'install', '--python', PY, 'paddlepaddle-gpu==3.2.0', '-i', 'https://www.paddlepaddle.org.cn/packages/stable/cu118/'], check=True)
SOURCE = WORK / 'PaddleOCR'
if not (SOURCE / '.git').is_dir():
    subprocess.run(['git', 'clone', 'https://github.com/PaddlePaddle/PaddleOCR.git', str(SOURCE)], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'checkout', '--detach', 'b03f46425e8ff4442b268ce449e3eef758146cd4'], check=True)
subprocess.run([uv, 'pip', 'install', '--python', PY, '-r', str(SOURCE / 'requirements.txt')], check=True)
subprocess.run([uv, 'pip', 'install', '--python', PY, 'numpy==1.26.4'], check=True)
subprocess.run([PY, '-c', 'import paddle; paddle.set_device("gpu:0"); assert paddle.device.is_compiled_with_cuda(); print("Paddle CUDA:", paddle.__version__, paddle.device.get_device())'], check=True)


In [ ]:
import collections, contextlib, hashlib, json, random, re, zipfile
from pathlib import PurePosixPath

# Override these paths if you attach several versions of the inputs.
IMAGE_ZIP = None
IMAGE_DIR = None  # Optional extracted-image directory, e.g. /kaggle/input/t22222
ANNOTATIONS_JSON = None
COCO_ZIP = None
COCO_DIR = None  # Optional extracted COCO root containing train/, valid/, test/.
SEED = 42
# Explicit mapping supplied with this dataset: c -> س, e -> ع, h -> ه, r -> ر.
LETTER_MAP = {'a':'ا','b':'ب','c':'س','d':'د','e':'ع','f':'ف','g':'ج','h':'ه',
               'l':'ل','m':'م','n':'ن','q':'ق','r':'ر','s':'ص','t':'ط','w':'و','y':'ي'}
COCO_LETTERS = {'Taa':'ط','ain':'ع','alif':'ا','baa':'ب','daal':'د','faa':'ف',
                'haa':'ه','jeem':'ج','laam':'ل','meem':'م','noon':'ن','qaaf':'ق',
                'raa':'ر','saad':'ص','seen':'س','waw':'و','yaa':'ي'}
VOCAB = '0123456789' + ''.join(dict.fromkeys(LETTER_MAP.values()))

def select_file(name, override, required=True):
    if override:
        path = pathlib.Path(override)
        if not path.is_file(): raise FileNotFoundError(path)
        return path
    matches = sorted(INPUT.rglob(name))
    if not matches and not required: return None
    if len(matches) != 1: raise RuntimeError(f'Expected exactly one {name} under /kaggle/input; found {matches}')
    return matches[0]

image_zip = select_file('T2.zip', IMAGE_ZIP, required=False)
coco_zip = select_file('egyptian_car_plates_filtered_no_forbidden_classes_or_0.zip', COCO_ZIP, required=False)
if coco_zip is None:
    coco_roots = ([pathlib.Path(COCO_DIR)] if COCO_DIR else
                  sorted({p.parent.parent for p in INPUT.rglob('_annotations.coco.json')
                          if p.parent.name == 'train'}))
    if len(coco_roots) != 1: raise RuntimeError(f'Expected one extracted COCO dataset root; found {coco_roots}')
    coco_root = coco_roots[0]
    if not coco_root.is_dir(): raise FileNotFoundError(coco_root)
json_matches = sorted(INPUT.rglob('ocr_ocr_labeling.json'))
if ANNOTATIONS_JSON or json_matches:
    annotation_path = select_file('ocr_ocr_labeling.json', ANNOTATIONS_JSON)
    annotations = json.loads(annotation_path.read_text(encoding='utf-8-sig'))
else:
    if image_zip is None:
        raise RuntimeError('Add ocr_ocr_labeling.json as a Kaggle input; T2.zip was unpacked, so the JSON must be attached separately')
    with zipfile.ZipFile(image_zip) as z:
        matches = [n for n in z.namelist() if PurePosixPath(n).name == 'ocr_ocr_labeling.json']
        if len(matches) != 1: raise RuntimeError('Provide ocr_ocr_labeling.json in T2.zip or as a separate Kaggle input')
        annotations = json.loads(z.read(matches[0]).decode('utf-8-sig'))
if not isinstance(annotations, dict) or not isinstance(annotations.get('data'), list):
    raise ValueError('Expected labelset OCR JSON with a data array')

ROOT = WORK / 'direct-ppocr' / MODEL
ROOT.mkdir(parents=True, exist_ok=True)
images = ROOT / 'images'
images.mkdir(exist_ok=True)
rows = []
def safe_member(raw):
    part = PurePosixPath(raw.replace('\\', '/'))
    if part.is_absolute() or '..' in part.parts or re.match(r'^[A-Za-z]:', raw):
        raise ValueError(f'Unsafe dataset member: {raw}')
    return part

with zipfile.ZipFile(image_zip) if image_zip else contextlib.nullcontext() as z:
    members = {}
    wanted = {PurePosixPath(str(item['fileName']).replace('\\', '/')).name for item in annotations['data']}
    if z:
        for info in z.infolist():
            raw = info.filename.replace('\\', '/')
            part = PurePosixPath(raw)
            if part.is_absolute() or '..' in part.parts or raw.startswith('/') or re.match(r'^[A-Za-z]:', raw):
                raise ValueError(f'Unsafe ZIP member: {raw}')
            if (info.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError(f'Symlink in ZIP: {raw}')
            if info.is_dir() or part.name not in wanted: continue
            if part.name in members: raise ValueError(f'Duplicate image basename: {part.name}')
            members[part.name] = info
    else:
        image_dir = pathlib.Path(IMAGE_DIR) if IMAGE_DIR else INPUT
        if not image_dir.is_dir(): raise FileNotFoundError(image_dir)
        for path in image_dir.rglob('*'):
            if path.name not in wanted or not path.is_file(): continue
            if path.is_symlink(): raise ValueError(f'Symlink in image directory: {path}')
            if path.name in members: raise ValueError(f'Duplicate image basename: {path.name}; set IMAGE_DIR')
            members[path.name] = path
        print('Reading extracted images from', image_dir)
    seen = set()
    for item in annotations['data']:
        name = PurePosixPath(str(item['fileName']).replace('\\', '/')).name
        if name in seen: raise ValueError(f'Duplicate annotation: {name}')
        seen.add(name)
        if name not in members: raise ValueError(f'Missing old dataset image: {name}')
        regions = item.get('regionLabel') or []
        if not regions: continue  # Unannotated images are never negative recognition labels.
        if item.get('set') not in ('train', 'test'):
            raise ValueError(f'Annotated image without train/test set: {name}')
        chars = []
        for part in sorted(regions, key=lambda p: (float(p['x'])+float(p.get('width',0))/2, float(p['y']))):
            code = str(part.get('className', '')).lower()
            if len(code) != 1 or (code not in LETTER_MAP and code not in '0123456789'):
                raise ValueError(f'Unsupported OCR class {code!r} in {name}')
            chars.append(LETTER_MAP.get(code, code))
        label = ''.join(chars)
        digits = ''.join(c for c in label if c in '0123456789')
        letters = ''.join(c for c in label if c not in '0123456789')
        if label != digits + letters or not (1 <= len(digits) <= 4 and 1 <= len(letters) <= 3):
            raise ValueError(f'Unexpected plate sequence in {name}: {label}')
        content = z.read(members[name]) if z else members[name].read_bytes()
        destination = images / 'old' / name
        destination.parent.mkdir(exist_ok=True)
        destination.write_bytes(content)
        match = re.match(r'^trk_(\d+)_', name)
        group = 'old:trk:' + match.group(1) if match else 'old:file:' + name
        rows.append({'name':'old/'+name, 'label':label, 'split':item['set'],
                     'group':group, 'hash':hashlib.sha256(content).hexdigest()})
with zipfile.ZipFile(coco_zip) if coco_zip else contextlib.nullcontext() as z:
    if z:
        members = {}
        for info in z.infolist():
            part = safe_member(info.filename)
            if (info.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError(f'Symlink in ZIP: {info.filename}')
            if info.is_dir(): continue
            key = str(part)
            if key in members: raise ValueError(f'Duplicate COCO member: {key}')
            members[key] = info
    for source_split, split in (('train','train'),('valid','validation'),('test','test')):
        annotation_name = source_split + '/_annotations.coco.json'
        if z:
            if annotation_name not in members: raise FileNotFoundError(annotation_name)
            coco = json.loads(z.read(members[annotation_name]).decode('utf-8-sig'))
        else:
            coco = json.loads((coco_root / annotation_name).read_text(encoding='utf-8-sig'))
        categories = {c['id']:c['name'] for c in coco['categories']}
        if len(categories) != len(coco['categories']): raise ValueError('Duplicate COCO category ID')
        annotations = collections.defaultdict(list)
        for annotation in coco['annotations']:
            annotations[annotation['image_id']].append(annotation)
        seen_ids, seen_names = set(), set()
        for image in coco['images']:
            image_id = image['id']
            name = image['file_name']
            if image_id in seen_ids or name in seen_names: raise ValueError(f'Duplicate COCO image: {source_split}/{name}')
            seen_ids.add(image_id)
            seen_names.add(name)
            if safe_member(name).name != name: raise ValueError(f'COCO image must be a basename: {name}')
            parts = annotations.pop(image_id, [])
            if not parts: continue  # Recognition cannot train on images without character labels.
            characters = []
            for annotation in sorted(parts, key=lambda a: (float(a['bbox'][0])+float(a['bbox'][2])/2, float(a['bbox'][1]))):
                category = categories[annotation['category_id']]
                if category not in COCO_LETTERS and category not in '123456789':
                    raise ValueError(f'Unsupported COCO class {category!r} in {source_split}/{name}')
                characters.append(COCO_LETTERS.get(category, category))
            label = ''.join(characters)
            if not 1 <= len(label) <= 7: raise ValueError(f'Unexpected COCO label length in {source_split}/{name}: {label}')
            member_name = source_split + '/' + name
            if z:
                if member_name not in members: raise FileNotFoundError(member_name)
                content = z.read(members[member_name])
            else:
                path = coco_root / member_name
                if path.is_symlink(): raise ValueError(f'Symlink in COCO dataset: {path}')
                content = path.read_bytes()
            destination = images / 'new' / source_split / name
            destination.parent.mkdir(parents=True, exist_ok=True)
            destination.write_bytes(content)
            origin = name.split('.rf.')[0]
            rows.append({'name':'new/'+member_name, 'label':label, 'split':split,
                         'group':'new:origin:'+origin, 'hash':hashlib.sha256(content).hexdigest()})
        if annotations: raise ValueError(f'COCO annotations reference unknown images in {source_split}: {list(annotations)[:5]}')
if not any(r['name'].startswith('old/') and r['split']=='test' for r in rows):
    raise ValueError('Expected labeled old test images')

# Union old tracks, new source images, and identical bytes across both datasets.
parents = {r['name']:r['name'] for r in rows}
def find(n):
    while parents[n] != n:
        parents[n] = parents[parents[n]]
        n = parents[n]
    return n
def union(a,b): parents[find(a)] = find(b)
for key in ('group','hash'):
    first = {}
    for r in rows:
        value = r[key]
        if value in first: union(r['name'],first[value])
        else: first[value] = r['name']
test_groups = {find(r['name']) for r in rows if r['split']=='test'}
if any(find(r['name']) in test_groups for r in rows if r['split']=='validation'):
    raise ValueError('Validation and test contain overlapping source images')
held_out = test_groups | {find(r['name']) for r in rows if r['split']=='validation'}
usable = [r for r in rows if r['split']=='train' and find(r['name']) not in held_out]
excluded = sum(r['split']=='train' for r in rows) - len(usable)
split_rows = {'train':usable,
              'validation':[r for r in rows if r['split']=='validation'],
              'test':[r for r in rows if r['split']=='test']}
for split, values in split_rows.items():
    if not values: raise ValueError(f'No labeled {split} images')
    (ROOT / f'{split}.txt').write_text(''.join(f'images/{r["name"]}\t{r["label"]}\n' for r in values), encoding='utf-8')
(ROOT / 'plate_dict.txt').write_text(''.join(c+'\n' for c in VOCAB), encoding='utf-8')
(ROOT / 'split.json').write_text(json.dumps({'excluded_train_overlapping_held_out':excluded,
     'splits':{s:[r['name'] for r in values] for s,values in split_rows.items()}}, indent=2), encoding='utf-8')
print('Dictionary:', VOCAB, 'Samples:', {s:len(v) for s,v in split_rows.items()},
       'train images sharing track/origin/hash with validation or test excluded:', excluded)
print('Sample labels:', [(r['name'],r['label']) for r in split_rows['train'][:5]])


In [ ]:
"""Self-contained progress parsing embedded in each direct-dataset Kaggle notebook."""
import csv
import hashlib
import json
import re
import zipfile
from contextlib import nullcontext
from pathlib import Path


_EPOCH = re.compile(r"epoch:\s*\[(\d+)/(\d+)\]")
_NUMBER = r"([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)"
_CHECKPOINT_FILES = {'latest.pdparams', 'latest.pdopt', 'latest.states',
                     'best_accuracy.pdparams', 'best_accuracy.pdopt', 'best_accuracy.states'}
_REPORT_FILES = {'training.log', 'epoch-metrics.csv', 'epoch-metrics.json', 'training-report.md'}


def find_resume(input_root, model, dataset_sha256, total_epochs):
    """Kaggle may mount a resume ZIP as a file or unpack it into a dataset folder."""
    root = Path(input_root)
    candidates = []
    for path in root.rglob('*.zip'):
        with zipfile.ZipFile(path) as archive:
            if 'resume.json' not in archive.namelist():
                continue  # Image or finished-model ZIP, not a resume input.
            if archive.namelist().count('resume.json') != 1:
                raise ValueError(f'Duplicate resume manifest in {path}')
            manifest = json.loads(archive.read('resume.json'))
        candidates.append((path, manifest))
    for path in root.rglob('resume.json'):
        candidates.append((path.parent, json.loads(path.read_text(encoding='utf-8'))))
    if not candidates:
        if any(root.rglob('latest.pdparams')):
            raise ValueError('Found extracted training weights but no resume.json; attach the complete resume ZIP')
        return None
    compatible = [(path, value) for path, value in candidates if value.get('model') == model]
    if not compatible:
        raise ValueError(f'Resume input belongs to a different model; expected {model}')
    for path, manifest in compatible:
        if manifest.get('dataset_sha256') != dataset_sha256:
            raise ValueError(f'Resume dataset fingerprint mismatch: {path}')
        epoch = int(manifest.get('completed_epochs', 0))
        if not 0 < epoch < total_epochs:
            raise ValueError(f'Resume epoch must be between 1 and {total_epochs - 1}: {path}')
    latest_epoch = max(int(value['completed_epochs']) for _, value in compatible)
    latest = [(path, value) for path, value in compatible if int(value['completed_epochs']) == latest_epoch]
    if any(value.get('files') != latest[0][1].get('files') for _, value in latest[1:]):
        raise ValueError(f'Conflicting resume inputs at epoch {latest_epoch}')
    # Prefer an attached ZIP when Kaggle exposes both the archive and its extraction.
    return next(((path, value) for path, value in latest if path.is_file()), latest[0])


def restore_resume(source, manifest, root):
    """Verify every checkpoint/report hash before using it for continuation."""
    source, root = Path(source), Path(root)
    files = manifest.get('files')
    if not isinstance(files, dict) or not {'latest.pdparams','latest.pdopt','latest.states'} <= files.keys():
        raise ValueError('Resume checkpoint is incomplete')
    if set(files) - (_CHECKPOINT_FILES | _REPORT_FILES):
        raise ValueError('Resume contains an unexpected file')
    with zipfile.ZipFile(source) if source.is_file() else nullcontext(None) as archive:
        if archive:
            names = archive.namelist()
            if any(names.count(name) != 1 for name in files):
                raise ValueError('Resume ZIP has missing or duplicate checkpoint members')
        for name, digest in files.items():
            if archive:
                data = archive.read(name)
            else:
                path = source / name
                if not path.is_file() or path.is_symlink():
                    raise ValueError(f'Resume file is missing or unsafe: {name}')
                data = path.read_bytes()
            if hashlib.sha256(data).hexdigest() != digest:
                raise ValueError(f'Resume hash mismatch: {name}')
            dest = (root / 'training' if name in _CHECKPOINT_FILES else root) / name
            dest.parent.mkdir(parents=True, exist_ok=True)
            dest.write_bytes(data)


def training_epoch(line):
    match = _EPOCH.search(line)
    return int(match.group(1)) if match else None


def _field(line, name):
    match = re.search(r"\b" + re.escape(name) + r":\s*" + _NUMBER, line)
    return float(match.group(1)) if match else None


class WorkerLogFollower:
    """Read newly appended, complete UTF-8 lines without waiting for process exit."""

    def __init__(self, path):
        self.path = Path(path)
        self.offset = 0
        self.pending = ""

    def poll(self, final=False):
        if self.path.is_file():
            with self.path.open("r", encoding="utf-8", errors="replace") as handle:
                handle.seek(self.offset)
                self.pending += handle.read()
                self.offset = handle.tell()
        parts = self.pending.split("\n")
        self.pending = parts.pop()
        if final and self.pending:
            parts.append(self.pending)
            self.pending = ""
        return [part.rstrip("\r") for part in parts]


def parse_training_log(text):
    """Parse step-level loss and validation events from PaddleOCR rank-zero logs."""
    steps, evaluations, best_events = [], [], []
    epoch = None
    for line in text.splitlines():
        observed = training_epoch(line)
        if observed is not None:
            epoch = observed
            loss = _field(line, "loss")
            if loss is not None:
                steps.append({"epoch": epoch, "global_step": _field(line, "global_step"),
                              "loss": loss, "ctc_loss": _field(line, "CTCLoss"),
                              "nrtr_loss": _field(line, "NRTRLoss"), "lr": _field(line, "lr"),
                              "ips": _field(line, "ips")})
        if epoch is not None and "cur metric," in line:
            evaluations.append({"epoch": epoch, "acc": _field(line, "acc"),
                                "norm_edit_dis": _field(line, "norm_edit_dis"),
                                "fps": _field(line, "fps")})
        if epoch is not None and "best metric," in line:
            best_events.append({"epoch": epoch, "acc": _field(line, "acc"),
                                "best_epoch": _field(line, "best_epoch")})
    epochs = []
    for number in sorted({step["epoch"] for step in steps}):
        observed_steps = [step for step in steps if step["epoch"] == number]
        validation = [row for row in evaluations if row["epoch"] == number]
        best = [row for row in best_events if row["epoch"] == number]
        def mean(key):
            values = [step[key] for step in observed_steps if step[key] is not None]
            return sum(values) / len(values) if values else None
        epochs.append({"epoch": number, "reported_steps": len(observed_steps),
                       "train_loss_mean": mean("loss"), "ctc_loss_mean": mean("ctc_loss"),
                       "nrtr_loss_mean": mean("nrtr_loss"), "last_lr": observed_steps[-1]["lr"],
                       "validation_acc": validation[-1]["acc"] if validation else None,
                       "validation_norm_edit_dis": validation[-1]["norm_edit_dis"] if validation else None,
                       "validation_fps": validation[-1]["fps"] if validation else None,
                       "best_acc": best[-1]["acc"] if best else None,
                       "best_epoch": best[-1]["best_epoch"] if best else None})
    return {"epochs": epochs, "steps": steps, "validation_events": evaluations}


def parse_test_metrics(text):
    # tools/eval.py emits a checkpoint-metric section and then a separate
    # 'metric eval ***************' block. Only the latter is held-out accuracy.
    rows = text.split("metric eval ***************")
    if len(rows) < 2:
        return None
    section = rows[-1]
    return {name: _field(section, name) for name in ("acc", "norm_edit_dis", "fps")}


def _display(number, places=4):
    return "—" if number is None else f"{number:.{places}f}"


def write_training_report(root, model, counts, target, gpu_names, batch, test_log=None):
    root = Path(root)
    log = root / "training.log"
    parsed = parse_training_log(log.read_text(encoding="utf-8", errors="replace") if log.is_file() else "")
    test = parse_test_metrics(Path(test_log).read_text(encoding="utf-8", errors="replace")) if test_log else None
    result = {"model": model, "through_epoch": target, "gpu_names": list(gpu_names),
              "batch_size_per_gpu": batch, "samples": dict(counts), **parsed,
              "held_out_test": test}
    (root / "epoch-metrics.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    columns = ("epoch", "reported_steps", "train_loss_mean", "ctc_loss_mean",
               "nrtr_loss_mean", "last_lr", "validation_acc", "validation_norm_edit_dis",
               "validation_fps", "best_acc", "best_epoch")
    with (root / "epoch-metrics.csv").open("w", encoding="utf-8", newline="") as output:
        writer = csv.DictWriter(output, fieldnames=columns)
        writer.writeheader()
        writer.writerows(parsed["epochs"])
    lines = [f"# {model}: training performance through epoch {target}", "",
             f"GPUs: {', '.join(gpu_names)} | batch per GPU: {batch} | samples: " +
             ", ".join(f"{split}={count}" for split, count in counts.items()), "",
             "Loss averages use every reported training step (log_smooth_window=1). "
             "Validation columns show the last evaluation in each epoch; — means none was logged.", "",
             "| Epoch | Steps | Train loss | CTC loss | NRTR loss | LR (last) | Val acc | Val normalized edit similarity | Best acc |",
             "| ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |"]
    if parsed["epochs"] and parsed["epochs"][0]["epoch"] > 1:
        lines.insert(4, f"Earlier epoch logs were not included in this resume; metrics start at epoch {parsed['epochs'][0]['epoch']}.")
    for row in parsed["epochs"]:
        lines.append("| " + " | ".join([
            str(row["epoch"]), str(row["reported_steps"]), _display(row["train_loss_mean"]),
            _display(row["ctc_loss_mean"]), _display(row["nrtr_loss_mean"]),
            _display(row["last_lr"], places=8), _display(row["validation_acc"]),
            _display(row["validation_norm_edit_dis"]), _display(row["best_acc"]),
        ]) + " |")
    lines.extend(["", "Full per-step values, validation events and GPU provenance are in "
                  "`epoch-metrics.json`; the unabridged Paddle output is in `training.log`."])
    if test:
        lines.extend(["", "## Held-out test (epoch 60)", "",
                      f"Accuracy: {_display(test['acc'])} | normalized edit similarity: "
                      f"{_display(test['norm_edit_dis'])} | images/second: {_display(test['fps'])}"])
    else:
        lines.extend(["", "Held-out test evaluation is reserved for epoch 60."])
    (root / "training-report.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
    return result

import tarfile, urllib.request, pickle, threading, time

# Each Kaggle run trains one 20-epoch chunk. Add the last resume ZIP as a Kaggle
# input to continue to epoch 40 and then 60; keep both datasets and JSON unchanged.
CHUNK_EPOCHS = 20
TOTAL_EPOCHS = 60
BATCH_PER_GPU = 8 if MODEL != 'v5_server' else 4
PRETRAINED_URLS = {
    'v5_server': 'https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv5_server_rec_pretrained.pdparams',
    'v5_arabic_mobile': 'https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/arabic_PP-OCRv5_mobile_rec_pretrained.pdparams',
}
config = yaml.safe_load((SOURCE / CONFIG_PATH).read_text(encoding='utf-8'))
# NRTRLabelEncode appends start/end tokens and rejects labels of length >= max-1.
# Seven visible plate characters therefore require at least max_text_length=9.
max_text_length = max(len(r['label']) for split in split_rows.values() for r in split) + 2
config['Global'].update(use_gpu=True, save_model_dir=str(ROOT/'training'),
    character_dict_path=str(ROOT/'plate_dict.txt'), max_text_length=max_text_length, use_space_char=False,
    seed=SEED, distributed=False, save_epoch_step=TOTAL_EPOCHS+1, print_batch_step=1,
    log_smooth_window=1, eval_batch_step=[0, 1], eval_batch_epoch=1,
    cal_metric_during_train=False, use_amp=True, amp_level='O2',
    scale_loss=1024.0, use_dynamic_loss_scaling=True)
config['Architecture']['Head']['head_list'][1]['NRTRHead']['max_text_length'] = max_text_length
config['Optimizer']['lr'].update(learning_rate=5e-5, warmup_epoch=1)
config['Train']['sampler']['first_bs'] = BATCH_PER_GPU
for key, name in (('Train','train'),('Eval','validation')):
    config[key]['dataset'].update(data_dir=str(ROOT), label_file_list=[str(ROOT/f'{name}.txt')])
    config[key]['loader'].update(batch_size_per_card=BATCH_PER_GPU, num_workers=2)
config['Eval']['loader']['shuffle'] = False
config['Eval']['loader']['drop_last'] = False
# Combining plates would make artificial labels unlike the target plate images.
config['Train']['dataset']['transforms'] = [op for op in config['Train']['dataset']['transforms'] if 'RecConAug' not in op]

fingerprint = hashlib.sha256(json.dumps(sorted((r['name'],r['hash'],r['label'],r['split']) for r in rows),
    ensure_ascii=False).encode('utf-8')).hexdigest()
resume = find_resume(INPUT, MODEL, fingerprint, TOTAL_EPOCHS)
completed = 0
if resume:
    resume_source, manifest = resume
    completed = int(manifest['completed_epochs'])
    restore_resume(resume_source, manifest, ROOT)
    print('Resuming', MODEL, 'from epoch', completed, 'using', resume_source)
    config['Global']['checkpoints'] = str(ROOT/'training'/'latest')
    config['Global']['pretrained_model'] = ''
else:
    base = ROOT/'base.pdparams'
    if MODEL == 'v4_mobile':
        extracted = [path for path in sorted(INPUT.rglob('best_accuracy.pdparams'))
                     if 'arabic_PP-OCRv4_rec_train' in path.parts]
        if len(extracted) > 1: raise RuntimeError(f'Multiple extracted v4 checkpoints: {extracted}')
        if extracted:
            shutil.copyfile(extracted[0], base)
            print('Using extracted v4 checkpoint:', extracted[0])
        else:
            found = []
            for archive in sorted(INPUT.rglob('*.tar')):
                with tarfile.open(archive) as t:
                    members = [m for m in t.getmembers() if PurePosixPath(m.name).name == 'best_accuracy.pdparams' and m.isfile()]
                    if len(members) > 1: raise RuntimeError(f'Multiple v4 weights in {archive}')
                    if members: found.append((archive, members[0].name))
            if len(found) != 1:
                raise RuntimeError(f'Expected one v4 best_accuracy.pdparams inside a tar or extracted Kaggle input; found {found}')
            archive, member = found[0]
            with tarfile.open(archive) as t, t.extractfile(member) as src, base.open('wb') as dst:
                shutil.copyfileobj(src, dst)
            print('Using v4 checkpoint:', archive, member)
        # Verify every non-vocabulary tensor before training: the supplied
        # checkpoint is PP-OCRv4 MOBILE, not PP-OCRv4 server.
        checker = ROOT/'verify_v4.py'
        checker.write_text('''import sys, yaml, paddle
sys.path.insert(0, sys.argv[1])
from ppocr.modeling.architectures import build_model
c = yaml.safe_load(open(sys.argv[2], encoding='utf8'))
c['Architecture']['Head']['out_channels_list'] = {'CTCLabelDecode':28,'NRTRLabelDecode':31}
model = build_model(c['Architecture']).state_dict()
weights = paddle.load(sys.argv[3])
heads = {'head.ctc_head.fc.weight','head.ctc_head.fc.bias',
         'head.gtc_head.embedding.embedding.weight','head.gtc_head.tgt_word_prj.weight'}
bad = [k for k in set(weights)|set(model) if k not in heads and
       (k not in weights or k not in model or list(weights[k].shape)!=list(model[k].shape))]
if bad: raise RuntimeError('Incompatible v4 backbone/neck: '+str(bad[:12]))
print('Verified',len(model)-len(heads),'non-vocabulary parameters')
''', encoding='utf-8')
        subprocess.run([PY, str(checker), str(SOURCE), str(SOURCE/CONFIG_PATH), str(base)], check=True)
    else:
        urllib.request.urlretrieve(PRETRAINED_URLS[MODEL], base)
    config['Global']['pretrained_model'] = str(base)
    config['Global']['checkpoints'] = ''

target = min(TOTAL_EPOCHS, completed + CHUNK_EPOCHS)
config['Global']['epoch_num'] = target
effective = ROOT/'effective.yml'
effective.write_text(yaml.safe_dump(config, sort_keys=False, allow_unicode=True), encoding='utf-8')
previous_best = ROOT/'previous-best'
if resume and (ROOT/'training'/'best_accuracy.states').is_file():
    previous_best.mkdir(exist_ok=True)
    for path in (ROOT/'training').glob('best_accuracy.*'):
        shutil.copy2(path,previous_best/path.name)
gpu_names = subprocess.check_output(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],text=True).strip().splitlines()
gpu_count = len(gpu_names)
if not gpu_count: raise RuntimeError('No NVIDIA GPU found')
train_args = ['tools/train.py','-c',str(effective)]
if gpu_count > 1:
    logs = ROOT/'distributed-logs'/f'epoch-{completed+1}-{target}'
    logs.mkdir(parents=True,exist_ok=True)
    command = [PY,'-m','paddle.distributed.launch',f'--gpus={",".join(map(str,range(gpu_count)))}',f'--log_dir={logs}',*train_args]
else:
    command = [PY,*train_args]
print(f'Training {MODEL}: epochs {completed+1}-{target} on {gpu_count} GPU(s)',flush=True)
log_path = ROOT/'training.log'
counts = {split:len(values) for split,values in split_rows.items()}
reported = set()
current_epoch = None
def epoch_summary(number):
    if number in reported: return
    result = write_training_report(ROOT, MODEL, counts, number, gpu_names, BATCH_PER_GPU)
    row = next((row for row in result['epochs'] if row['epoch'] == number), None)
    if row:
        print(f"Epoch {number}/{target} complete | mean training loss: {_display(row['train_loss_mean'])} "
              f"| CTC: {_display(row['ctc_loss_mean'])} | NRTR: {_display(row['nrtr_loss_mean'])} "
              f"| validation acc: {_display(row['validation_acc'])} "
              f"| normalized edit similarity: {_display(row['validation_norm_edit_dis'])}", flush=True)
        reported.add(number)

def record(line, output):
    global current_epoch
    number = training_epoch(line)
    if number is not None and current_epoch is not None and number > current_epoch:
        output.flush()
        epoch_summary(current_epoch)
    if number is not None and number != current_epoch:
        current_epoch = number
        print(f'Epoch {number}/{target} started',flush=True)
    output.write(line + '\n')
    output.flush()
    print(line,flush=True)

process = subprocess.Popen(command,cwd=SOURCE,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                           text=True,errors='replace',bufsize=1)
if gpu_count == 1:
    with log_path.open('a',encoding='utf-8') as output:
        for line in process.stdout:
            record(line.rstrip('\r\n'), output)
    code = process.wait()
else:
    launcher_log = ROOT/'launcher.log'
    def read_launcher():
        with launcher_log.open('w',encoding='utf-8') as output:
            for line in process.stdout:
                output.write(line)
                output.flush()
                if any(word in line.lower() for word in ('traceback','error','failed')):
                    print('Launcher:',line.rstrip(),flush=True)
    reader = threading.Thread(target=read_launcher,daemon=True)
    reader.start()
    follower = WorkerLogFollower(logs/'workerlog.0')
    with log_path.open('a',encoding='utf-8') as output:
        while process.poll() is None:
            for line in follower.poll(): record(line, output)
            time.sleep(.2)
        for line in follower.poll(final=True): record(line, output)
    code = process.wait()
    reader.join(timeout=10)
    if not follower.path.is_file():
        print('Rank-zero log did not appear. Launcher output:',launcher_log.read_text(encoding='utf-8',errors='replace')[-12000:])
    if code:
        for worker in sorted(logs.glob('workerlog.*')):
            if worker.name != 'workerlog.0':
                print(f'--- {worker.name} (last 80 lines) ---')
                print('\n'.join(worker.read_text(encoding='utf-8',errors='replace').splitlines()[-80:]))
if current_epoch is not None:
    epoch_summary(current_epoch)
if code:
    print('--- training.log (last 80 lines) ---')
    print('\n'.join(log_path.read_text(encoding='utf-8',errors='replace').splitlines()[-80:]))
    raise RuntimeError(f'PaddleOCR training failed (exit code {code}); see rank-zero and launcher logs above')
report = write_training_report(ROOT, MODEL, counts, target, gpu_names, BATCH_PER_GPU)
def accuracy(state):
    with state.open('rb') as handle: return float(pickle.load(handle)['best_model_dict']['acc'])
old_state = previous_best/'best_accuracy.states'
new_state = ROOT/'training'/'best_accuracy.states'
if old_state.is_file() and (not new_state.is_file() or accuracy(old_state) > accuracy(new_state)):
    for path in previous_best.glob('best_accuracy.*'):
        shutil.copy2(path,ROOT/'training'/path.name)
    print('Retained the higher-accuracy checkpoint from the preceding chunk')
latest = ROOT/'training'/'latest'
for suffix in ('.pdparams','.pdopt','.states'):
    if not latest.with_suffix(suffix).is_file(): raise RuntimeError(f'Missing latest{suffix}')
resume_file = WORK/f'{MODEL}-resume-epoch-{target}.zip'
with zipfile.ZipFile(resume_file,'w',zipfile.ZIP_DEFLATED,allowZip64=True) as z:
    names = {'latest'+s:latest.with_suffix(s) for s in ('.pdparams','.pdopt','.states')}
    names.update({p.name:p for p in (ROOT/'training').glob('best_accuracy.*')
                  if p.suffix in {'.pdparams','.states'}})
    names.update({name:ROOT/name for name in ('training.log','epoch-metrics.csv',
                  'epoch-metrics.json','training-report.md')})
    manifest = {'model':MODEL,'completed_epochs':target,'dataset_sha256':fingerprint,
                'files':{n:hashlib.sha256(p.read_bytes()).hexdigest() for n,p in names.items()}}
    z.writestr('resume.json',json.dumps(manifest))
    for name,path in names.items(): z.write(path,name)
for name in ('training.log','epoch-metrics.csv','epoch-metrics.json','training-report.md'):
    destination = WORK/f'{MODEL}-epoch-{target}-{name}'
    shutil.copy2(ROOT/name,destination)
    print('Saved:',destination)
    display(FileLink(destination.name))
display(FileLink(resume_file.name))


In [ ]:
best = ROOT/'training'/'best_accuracy'
if not best.with_suffix('.pdparams').is_file(): raise RuntimeError('No validation best_accuracy checkpoint found')
test_config = yaml.safe_load(effective.read_text(encoding='utf-8'))
test_config['Eval']['dataset']['label_file_list'] = [str(ROOT/'test.txt')]
test_config['Global']['checkpoints'] = str(best)
test_config['Global']['pretrained_model'] = ''
test_yaml = ROOT/'test-effective.yml'
test_yaml.write_text(yaml.safe_dump(test_config,allow_unicode=True,sort_keys=False),encoding='utf-8')
if target == TOTAL_EPOCHS:
    with (ROOT/'test-metrics.txt').open('w',encoding='utf-8') as report:
        result = subprocess.run([PY,'tools/eval.py','-c',str(test_yaml)],cwd=SOURCE,
            stdout=report,stderr=subprocess.STDOUT,text=True)
    print((ROOT/'test-metrics.txt').read_text(encoding='utf-8',errors='replace')[-5000:])
    if result.returncode: raise RuntimeError('Held-out evaluation failed')
    write_training_report(ROOT,MODEL,counts,target,gpu_names,BATCH_PER_GPU,ROOT/'test-metrics.txt')
    shutil.copy2(ROOT/'training-report.md',WORK/f'{MODEL}-epoch-{target}-training-report.md')
    shutil.copy2(ROOT/'epoch-metrics.json',WORK/f'{MODEL}-epoch-{target}-epoch-metrics.json')
else:
    print('Held-out test will be evaluated only after epoch', TOTAL_EPOCHS)
inference = ROOT/'inference'
subprocess.run([PY,'tools/export_model.py','-c',str(effective),'-o',
    f'Global.pretrained_model={best}',f'Global.save_inference_dir={inference}',
    'Global.checkpoints=', 'Global.distributed=False'],cwd=SOURCE,check=True)
if not (inference/'inference.pdiparams').is_file(): raise RuntimeError('Inference export failed')
package = WORK/f'{MODEL}-fine-tuned.zip'
with zipfile.ZipFile(package,'w',zipfile.ZIP_DEFLATED,allowZip64=True) as z:
    for path in [ROOT/'effective.yml',ROOT/'test-effective.yml',ROOT/'plate_dict.txt',
                 ROOT/'split.json',ROOT/'training.log',ROOT/'epoch-metrics.csv',
                 ROOT/'epoch-metrics.json',ROOT/'training-report.md',
                 *sorted(p for p in (ROOT/'training').glob('best_accuracy.*')
                         if p.suffix in {'.pdparams','.states'}),
                 *sorted(inference.glob('*')),
                 *([ROOT/'launcher.log'] if (ROOT/'launcher.log').is_file() else []),
                 *([ROOT/'test-metrics.txt'] if target == TOTAL_EPOCHS else [])]:
        z.write(path,str(path.relative_to(ROOT)))
display(FileLink(package.name))
print('Download model ZIP, resume ZIP and reports from Kaggle Output → /kaggle/working (refresh if needed).')
